In [ ]:
"C:/Users/ibuba/Kuliah/Proyek-Akhir/Aplikasi/Dataset/hasil_ocr/evaluasi model.xlsx"

In [1]:
import difflib
import pandas as pd

MODEL_COLUMNS = ["pytesseract", "donut", "keras", "llm qwen"]
GT_COLUMN = "teks asli"
FILE_COLUMN = "Nama File"
MATCH_THRESHOLD = 0.35   # min similarity ratio to count a field as "found"
SKIP_VALUES = {"tidak ditemukan", "n/a", "-", ""}


def levenshtein(a, b):
    n, m = len(a), len(b)
    if n == 0:
        return m
    if m == 0:
        return n
    prev = list(range(m + 1))
    for i in range(1, n + 1):
        curr = [i] + [0] * m
        ai = a[i - 1]
        for j in range(1, m + 1):
            cost = 0 if ai == b[j - 1] else 1
            curr[j] = min(
                prev[j] + 1,          # deletion
                curr[j - 1] + 1,      # insertion
                prev[j - 1] + cost,   # substitution
            )
        prev = curr
    return prev[m]


def cer(ref, hyp):
    """Character Error Rate."""
    if len(ref) == 0:
        return 0.0 if len(hyp) == 0 else 1.0
    return levenshtein(ref, hyp) / len(ref)


def wer(ref, hyp):
    """Word Error Rate."""
    ref_words, hyp_words = ref.split(), hyp.split()
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else 1.0
    return levenshtein(ref_words, hyp_words) / len(ref_words)


def find_best_span(entity, text):
    if not isinstance(text, str) or not text.strip():
        return "", 0.0

    sm = difflib.SequenceMatcher(None, text, entity, autojunk=False)
    match = sm.find_longest_match(0, len(text), 0, len(entity))
    if match.size == 0:
        return "", 0.0

    margin = max(20, int(len(entity) * 1.5))
    start = max(0, match.a - margin)
    end = min(len(text), match.a + len(entity) + margin)
    local_words = text[start:end].split()

    n_words = max(1, len(entity.split()))
    best_ratio, best_span = 0.0, ""

    for size in range(max(1, n_words - 1), n_words + 3):
        for s in range(0, max(1, len(local_words) - size + 1)):
            window = " ".join(local_words[s:s + size])
            ratio = difflib.SequenceMatcher(None, entity, window).ratio()
            if ratio > best_ratio:
                best_ratio = ratio
                best_span = window

    return best_span, best_ratio


def main(path):
    df = pd.read_excel(path, sheet_name="Sheet1")
    results = []

    for _, row in df.iterrows():
        filename = row.get(FILE_COLUMN)
        gt_raw = row.get(GT_COLUMN)
        if not isinstance(gt_raw, str) or not gt_raw.strip():
            continue

        entities = [e.strip() for e in gt_raw.split(",") if e.strip()]
        entities = [e for e in entities if e.lower() not in SKIP_VALUES]

        for model in MODEL_COLUMNS:
            model_text = row.get(model)
            model_text = model_text.lower() if isinstance(model_text, str) else ""

            for entity in entities:
                entity_l = entity.lower()
                span, ratio = find_best_span(entity_l, model_text)
                found = ratio >= MATCH_THRESHOLD

                results.append({
                    "file": filename,
                    "model": model,
                    "entity": entity,
                    "matched_span": span,
                    "match_ratio": round(ratio, 3),
                    "found": found,
                    "cer": cer(entity_l, span) if found else None,
                    "wer": wer(entity_l, span) if found else None,
                })

    detail_df = pd.DataFrame(results)

    summary = (
        detail_df.groupby("model")
        .agg(
            n_entities=("entity", "count"),
            n_found=("found", "sum"),
            found_rate=("found", "mean"),
            mean_cer=("cer", "mean"),
            mean_wer=("wer", "mean"),
        )
        .reset_index()
        .sort_values("mean_cer")
    )

    detail_df.to_excel("field_level_cer_wer_detail.xlsx", index=False)
    summary.to_excel("field_level_cer_wer_summary.xlsx", index=False)

    print(summary.to_string(index=False))
    print("\nDetail per entitas -> field_level_cer_wer_detail.xlsx")
    print("Ringkasan per model -> field_level_cer_wer_summary.xlsx")


if __name__ == "__main__":
    main("C:/Users/ibuba/Kuliah/Proyek-Akhir/Aplikasi/Dataset/hasil_ocr/evaluasi model_dengan_recall.xlsx")

      model  n_entities  n_found  found_rate  mean_cer  mean_wer
   llm qwen         170      168    0.988235  0.058223  0.135587
pytesseract         170      162    0.952941  0.224555  0.381278
      keras         170      160    0.941176  0.298221  0.464196
      donut         170       78    0.458824  0.474809  0.763675

Detail per entitas -> field_level_cer_wer_detail.xlsx
Ringkasan per model -> field_level_cer_wer_summary.xlsx
